# Module 4 Project Solution: Generative Vision, Diffusion, and Image-to-Image Translation


> **Colab setup:** This solution notebook uses only the default open-source Python stack available in Google Colab: NumPy, Matplotlib, and scikit-learn. No `pip install`, no API keys, and no external authorization are required.
>
> Running all cells produces the complete reference output, including plots and metrics.


## Project map
This project keeps generative vision lightweight: PCA gives a transparent latent space, while noising and denoising mirror diffusion concepts without heavy training.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

np.random.seed(7)
digits = load_digits()
X_img = digits.images / 16.0
X = X_img.reshape(len(X_img), -1)
y = digits.target
sample = X_img[0]


In [ ]:
T = 6
betas = np.linspace(0.02, 0.35, T)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

fig, axes = plt.subplots(1, T, figsize=(12, 2))
for t in range(T):
    noise = np.random.randn(*sample.shape)
    x_t = np.sqrt(alpha_bars[t]) * sample + np.sqrt(1 - alpha_bars[t]) * noise
    axes[t].imshow(np.clip(x_t, 0, 1), cmap="gray")
    axes[t].set_title(f"t={t+1}")
    axes[t].axis("off")
plt.suptitle("Forward noising process")
plt.show()

plt.plot(range(1, T+1), alpha_bars, marker="o")
plt.title("Signal retained across diffusion steps")
plt.xlabel("t")
plt.ylabel("alpha_bar")
plt.grid(True)
plt.show()


In [ ]:
pca = PCA(n_components=16, random_state=7)
Z = pca.fit_transform(X)
X_rec = pca.inverse_transform(Z)

fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i in range(8):
    axes[0, i].imshow(X_img[i], cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(X_rec[i].reshape(8, 8), cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("real")
axes[1, 0].set_ylabel("PCA rec")
plt.show()

plt.plot(np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.title("Cumulative explained variance")
plt.xlabel("components")
plt.ylabel("variance explained")
plt.grid(True)
plt.show()


In [ ]:
i, j = 0, 10
z1, z2 = Z[i], Z[j]
weights = np.linspace(0, 1, 8)
latents = np.array([(1-w) * z1 + w * z2 for w in weights])
imgs = pca.inverse_transform(latents)

plt.figure(figsize=(10, 2))
for k, im in enumerate(imgs):
    plt.subplot(1, 8, k+1)
    plt.imshow(im.reshape(8, 8), cmap="gray")
    plt.axis("off")
plt.suptitle("Latent interpolation")
plt.show()


In [ ]:
def mean_filter(image, k=3):
    pad = k // 2
    padded = np.pad(image, pad, mode="edge")
    out = np.zeros_like(image)
    for r in range(image.shape[0]):
        for c in range(image.shape[1]):
            patch = padded[r:r+k, c:c+k]
            out[r, c] = patch.mean()
    return out

noisy = np.clip(sample + 0.45 * np.random.randn(*sample.shape), 0, 1)
denoised = mean_filter(noisy, k=3)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(sample, cmap="gray"); axes[0].set_title("clean")
axes[1].imshow(noisy, cmap="gray"); axes[1].set_title("noisy")
axes[2].imshow(denoised, cmap="gray"); axes[2].set_title("denoised")
for ax in axes: ax.axis("off")
plt.show()

print("MSE noisy:", np.mean((sample - noisy)**2))
print("MSE denoised:", np.mean((sample - denoised)**2))
